## Google Ads

#### Useful links:
- [Get a Developer account](https://developers.facebook.com/async/registration/dialog/?src=default)
- [Google Ads API Documentation](https://www.facebook.com/ads/library/api/)
- [Pandas GBQ Documentation](https://pandas-gbq.readthedocs.io/en/latest/)
- [Google Ads Transparency](https://adstransparency.google.com/)
- [API Data Dictionary - README](https://storage.googleapis.com/ads-transparency-center/api-data/README.txt)

In [8]:
# requirements
%pip install pandas requests python-dotenv pandas-gbq tqdm

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import pandas as pd
import pandas_gbq
import pickle
import requests
from dotenv import load_dotenv
from datetime import datetime, timezone
from time import sleep

In [2]:
# environment variables
load_dotenv()
PROJECT_ID = os.getenv("PROJECT_ID")


This notebook assumes that no authentication other than project_id is necessary. If needed, please refer to [Pandas GBQ - Authentication](https://pandas-gbq.readthedocs.io/en/latest/howto/authentication.html#default-authentication-methods)

#### Google Ads Tables

**creative_stats:** contains information about advertisers that served ads in the European Economic Area or Turkey: their legal name, verification status, disclosed name, and location. It also includes ad specific information: impression ranges per region (including aggregate impressions for the European Economic Area), first shown and last shown dates, which criteria were used in audience selection, the format of the ad, the ad topic and whether the ad is funded by Google Ad Grants program. A link to the ad in the Google Ads Transparency Center is also provided.

**The removed_creative_stats:** contains information about ads that served in the European Economic Area that Google removed: where and why they were removed and per-region information on when they served. The removed_creative_stats table also contains a link to the Google Ads Transparency Center for the removed ad.

Both datasets are also available for download in [Google Storage](https://console.cloud.google.com/storage/browser/ads-transparency-center/api-data;tab=objects?prefix=&forceOnObjectsSortingFiltering=false)

In [3]:
creative_stats_tb = "bigquery-public-data.google_ads_transparency_center.creative_stats"
removed_creative_stats_tb = "bigquery-public-data.google_ads_transparency_center.removed_creative_stats"

In [4]:
# Count creative stats on Brazil
sql_query_br_cs = f"""
SELECT
  COUNT(*) AS line_count
FROM
  {creative_stats_tb} AS CS,
  CS.region_stats AS RS
WHERE
  RS.region_code = "BR" ;
"""

# Count remove creative stats on Brazil
sql_query_br_rcs = f"""
SELECT
  COUNT(*) AS line_count
FROM
  {removed_creative_stats_tb} AS CS,
  CS.region_stats AS RS
WHERE
  RS.region_code = "BR" ;
"""

#### API Connection

In [5]:
def query_api(project_id, select):
    return pandas_gbq.read_gbq(select, project_id=project_id)

#### Let's see how many rows we can get for ads displayed in Brazil

In [47]:
query_api(PROJECT_ID, sql_query_br_cs)

Downloading: 100%|█████████████████████████████████████████████████████████████|


,line_count
0,0


In [48]:
query_api(PROJECT_ID, sql_query_br_rcs)

Downloading: 100%|█████████████████████████████████████████████████████████████|


,line_count
0,0


As we can see, there are no creative stats available on Brazil, removed or otherwise.

The following questions are going to be answered based on another region were there is data available

In [49]:
# Count creative stats on France
sql_count_fr_cs = f"""
SELECT
  COUNT(*) AS line_count
FROM
  {creative_stats_tb} AS CS,
  CS.region_stats AS RS
WHERE
  RS.region_code = "FR" ;
"""

In [50]:
query_api(PROJECT_ID, sql_count_fr_cs)

Downloading: 100%|█████████████████████████████████████████████████████████████|


,line_count
0,35152334


### Let's sample some lines from the dataset tables
As a "SELECT *" query can be quite costly in terms of our monthly free quota for consulting public BigQuery datasets, to refrain from making unnecessary BigQuery requests, let's get samples from both publicly available creative stats tables and use them to answer questions where possible

In [6]:
# Get 100 lines of creative stats on France
sql_query_fr_cs = f"""
SELECT
  *
FROM
  {creative_stats_tb} AS CS,
  CS.region_stats AS RS
WHERE
  RS.region_code = "FR"
LIMIT
  100;
"""

df_creative_stats = query_api(project_id=PROJECT_ID, select=sql_query_fr_cs)
df_creative_stats

Downloading: 100%|█████████████████████████████████████████████████████████████|


,advertiser_id,creative_id,creative_page_url,ad_format_type,advertiser_disclosed_name,advertiser_legal_name,advertiser_location,advertiser_verification_status,region_stats,audience_selection_approach_info,...,ad_funded_by,region_code,first_shown,last_shown,times_shown_end_date,times_shown_lower_bound,times_shown_upper_bound,times_shown_start_date,times_shown_availability_date,surface_serving_stats
0,AR07993695893255618561,CR13395191900240084993,https://adstransparency.google.com/advertiser/...,TEXT,"Bons Websistemes, SL","Bons Websistemes, SL",AD,VERIFIED,"[{'region_code': 'AT', 'first_shown': '2023-03...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2023-03-01,2025-01-16,2025-01-16,0,1000,2023-03-08,None,{'surface_serving_stats': [{'surface': 'YOUTUB...
1,AR18080335191007559681,CR11361509887558287361,https://adstransparency.google.com/advertiser/...,TEXT,GRANJA ROCA GRUP SA,GRANJA ROCA GRUP SA,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-04-29,2025-09-01,2025-08-13,1000,2000,2025-04-29,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
2,AR10967695351116464129,CR06006986872738283521,https://adstransparency.google.com/advertiser/...,IMAGE,HILL SMART,HILL SMART,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-1...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,...,,FR,2025-11-09,2025-11-10,None,<NA>,<NA>,None,2026-02-07,None
3,AR02603609312274153473,CR15958400774443106305,https://adstransparency.google.com/advertiser/...,TEXT,JR NETMEDIA,JR NETMEDIA,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-04-01,2025-05-22,2025-05-22,0,1000,2025-04-01,None,None
4,AR12582294091645059073,CR17218540554137108481,https://adstransparency.google.com/advertiser/...,VIDEO,MAJESTICS BUSINESS SL,MAJESTICS BUSINESS SL,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2023-1...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2023-12-30,2025-04-21,2025-04-21,300000,350000,2023-12-30,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,AR04374491158106079233,CR02245881606449397761,https://adstransparency.google.com/advertiser/...,IMAGE,MMG Digital,MMG Digital,AE,VERIFIED,"[{'region_code': 'DE', 'first_shown': '2025-07...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-07-30,2025-08-08,2025-08-08,0,1000,2025-07-30,None,None
96,AR04374491158106079233,CR15797212163652714497,https://adstransparency.google.com/advertiser/...,IMAGE,MMG Digital,MMG Digital,AE,VERIFIED,"[{'region_code': 'AT', 'first_shown': '2025-05...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2024-11-23,2025-05-08,2025-05-08,400000,450000,2024-11-23,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
97,AR04374491158106079233,CR05382319338785079297,https://adstransparency.google.com/advertiser/...,IMAGE,MMG Digital,MMG Digital,AE,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-08-11,2025-08-11,2025-08-11,0,1000,2025-08-11,None,None
98,AR04374491158106079233,CR08941142571829166081,https://adstransparency.google.com/advertiser/...,IMAGE,MMG Digital,MMG Digital,AE,VERIFIED,"[{'region_code': 'BG', 'first_shown': '2025-08...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-08-24,2025-11-09,None,<NA>,<NA>,None,2025-11-22,None


In [7]:
# Get 100 lines of removed creative stats on France
sql_query_fr_rcs = f"""
SELECT
  *
FROM
  {removed_creative_stats_tb} AS CS,
  CS.region_stats AS RS
WHERE
  RS.region_code = "FR"
LIMIT
  100;
"""

df_removed_creative_stats = query_api(project_id=PROJECT_ID, select=sql_query_fr_rcs)
df_removed_creative_stats

Downloading: 100%|█████████████████████████████████████████████████████████████|


,creative_page_url,region_stats,audience_selection_approach_info,disapproval,region_code,first_shown,last_shown,times_shown_end_date,times_shown_lower_bound,times_shown_upper_bound,times_shown_start_date,times_shown_availability_date,surface_serving_stats
0,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ...",FR,2025-05-08,2025-05-08,2025-05-08,0,1000,2025-05-08,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
1,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'DE', 'first_shown': '2025-10...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,"[{'removal_reason': 'Misrepresentation', 'viol...",FR,2025-10-07,2025-10-07,None,<NA>,<NA>,None,2026-01-05,None
2,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'AT', 'first_shown': '2025-08...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Account suspension', 'vio...",FR,2025-08-29,2025-08-29,None,<NA>,<NA>,None,2025-11-27,None
3,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'CZ', 'first_shown': '2024-12...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",[{'removal_reason': 'Dating and Companionship'...,FR,2024-12-08,2024-12-28,2024-12-28,0,1000,2024-12-08,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
4,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Abusing the ad network', ...",FR,2025-05-01,2025-05-01,2025-05-01,0,1000,2025-05-01,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'DE', 'first_shown': '2024-11...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...","[{'removal_reason': 'Account suspension', 'vio...",FR,2024-11-24,2024-12-05,2024-12-05,0,1000,2024-11-24,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
96,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'DE', 'first_shown': '2025-03...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,"[{'removal_reason': 'Account suspension', 'vio...",FR,2025-03-16,2025-03-16,2025-03-16,0,1000,2025-03-16,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
97,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'BG', 'first_shown': '2025-04...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,"[{'removal_reason': 'Account suspension', 'vio...",FR,2025-04-01,2025-04-02,2025-04-02,0,1000,2025-04-01,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
98,https://adstransparency.google.com/removed/cre...,"[{'region_code': 'BE', 'first_shown': '2024-12...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,"[{'removal_reason': 'Account suspension', 'vio...",FR,2024-12-12,2025-02-08,2025-02-08,0,1000,2024-12-12,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."


## (Optional)
We might want to save the resulting samples on disk in case we need them later (also to avoid making more "SELECT *" requests to BigQuery).

In [1]:
# Writing on disk
with open('sample-fr-cs.pickle', 'wb') as f:
    pickle.dump(df_creative_stats, f)

with open('sample-fr-rcs.pickle', 'wb') as f:
    pickle.dump(df_removed_creative_stats, f)

In [2]:
# Reading from disk
with open('sample-fr-cs.pickle', 'rb') as f:
    df_creative_stats = pickle.load(f)

with open('sample-fr-rcs.pickle', 'rb') as f:
    df_removed_creative_stats = pickle.load(f)

### Special Criteria

**SC1: Does the platform provide an API to access its ad repository and extract data on advertising content?**

*This item verifies whether the platform provides an ad repository API with at least one endpoint for programmatically extracting advertising data. Full availability is confirmed when the API returns information on ads across all categories. The assessment should confirm that the endpoint allows the retrieval and storage of ad data without requiring privileged or internal access beyond standard developer registration.*


In [8]:
df_creative_stats

,advertiser_id,creative_id,creative_page_url,ad_format_type,advertiser_disclosed_name,advertiser_legal_name,advertiser_location,advertiser_verification_status,region_stats,audience_selection_approach_info,...,ad_funded_by,region_code,first_shown,last_shown,times_shown_end_date,times_shown_lower_bound,times_shown_upper_bound,times_shown_start_date,times_shown_availability_date,surface_serving_stats
0,AR07993695893255618561,CR13395191900240084993,https://adstransparency.google.com/advertiser/...,TEXT,"Bons Websistemes, SL","Bons Websistemes, SL",AD,VERIFIED,"[{'region_code': 'AT', 'first_shown': '2023-03...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2023-03-01,2025-01-16,2025-01-16,0,1000,2023-03-08,None,{'surface_serving_stats': [{'surface': 'YOUTUB...
1,AR18080335191007559681,CR11361509887558287361,https://adstransparency.google.com/advertiser/...,TEXT,GRANJA ROCA GRUP SA,GRANJA ROCA GRUP SA,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-04-29,2025-09-01,2025-08-13,1000,2000,2025-04-29,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
2,AR10967695351116464129,CR06006986872738283521,https://adstransparency.google.com/advertiser/...,IMAGE,HILL SMART,HILL SMART,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-1...",{'demographic_info': 'CRITERIA_INCLUDED_AND_EX...,...,,FR,2025-11-09,2025-11-10,None,<NA>,<NA>,None,2026-02-07,None
3,AR02603609312274153473,CR15958400774443106305,https://adstransparency.google.com/advertiser/...,TEXT,JR NETMEDIA,JR NETMEDIA,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-04-01,2025-05-22,2025-05-22,0,1000,2025-04-01,None,None
4,AR12582294091645059073,CR17218540554137108481,https://adstransparency.google.com/advertiser/...,VIDEO,MAJESTICS BUSINESS SL,MAJESTICS BUSINESS SL,AD,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2023-1...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2023-12-30,2025-04-21,2025-04-21,300000,350000,2023-12-30,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,AR04374491158106079233,CR02245881606449397761,https://adstransparency.google.com/advertiser/...,IMAGE,MMG Digital,MMG Digital,AE,VERIFIED,"[{'region_code': 'DE', 'first_shown': '2025-07...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-07-30,2025-08-08,2025-08-08,0,1000,2025-07-30,None,None
96,AR04374491158106079233,CR15797212163652714497,https://adstransparency.google.com/advertiser/...,IMAGE,MMG Digital,MMG Digital,AE,VERIFIED,"[{'region_code': 'AT', 'first_shown': '2025-05...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2024-11-23,2025-05-08,2025-05-08,400000,450000,2024-11-23,None,"{'surface_serving_stats': [{'surface': 'PLAY',..."
97,AR04374491158106079233,CR05382319338785079297,https://adstransparency.google.com/advertiser/...,IMAGE,MMG Digital,MMG Digital,AE,VERIFIED,"[{'region_code': 'EEA', 'first_shown': '2025-0...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-08-11,2025-08-11,2025-08-11,0,1000,2025-08-11,None,None
98,AR04374491158106079233,CR08941142571829166081,https://adstransparency.google.com/advertiser/...,IMAGE,MMG Digital,MMG Digital,AE,VERIFIED,"[{'region_code': 'BG', 'first_shown': '2025-08...","{'demographic_info': 'CRITERIA_INCLUDED', 'geo...",...,,FR,2025-08-24,2025-11-09,None,<NA>,<NA>,None,2025-11-22,None


**SC3: Can data from both active and inactive ads be extracted?**

*This item verifies whether the platform allows the extraction of ad data through either the GUI or the API, from at least one day after publication to at least one year prior. Full availability is confirmed when both active and inactive ad data are delivered across all ad categories. The assessment should test the interface and endpoints to confirm whether both active and inactive ads can be retrieved.*


In [10]:
print(f'Now {datetime.now(tz=timezone.utc).isoformat()}')
print(f'Most recent creative show date {df_creative_stats.last_shown.max()}')
print(f'Least recent creative show date {df_creative_stats.last_shown.min()}')

Now 2025-11-12T04:21:22.127083+00:00
Most recent creative show date 2025-11-10
Least recent creative show date 2024-08-12


In [ ]:
# Even though the above cell suffices for proving the point. This would be more accurate.
# This cell won't be run right now because of free quota limits, as comparison to the datetime.now() would require another data retrieval.

print(f'Now {datetime.now(tz=timezone.utc).isoformat()}')

# Iterate through the region_stats column to find the max and min dates
all_last_shown_dates = [
    region_stats['last_shown']
    for creative_region_stats in df_creative_stats.region_stats
    for region_stats in creative_region_stats
]

max_date = max(all_last_shown_dates)
min_date = min(all_last_shown_dates)

print(f'Most recent creative show date {max_date}')
print(f'Least recent creative show date {min_date}')

### ACCESSIBILITY


**OC3: Can the requested data be extracted directly from the ad repository response?**

This item verifies whether the platform ad repository returns structured data on ad creatives and authorship directly in the response, rather than providing a link that redirects to the data. Audiovisual media files and data (e.g., images, videos, and audio) should not be considered when assessing this item. The assessment should examine sample data responses from both the ad repository GUI and API to confirm that the requested public data is included in the returned payload.*


In [11]:
df_creative_stats.loc[0]

advertiser_id                                                  AR07993695893255618561
creative_id                                                    CR13395191900240084993
creative_page_url                   https://adstransparency.google.com/advertiser/...
ad_format_type                                                                   TEXT
advertiser_disclosed_name                                        Bons Websistemes, SL
advertiser_legal_name                                            Bons Websistemes, SL
advertiser_location                                                                AD
advertiser_verification_status                                               VERIFIED
region_stats                        [{'region_code': 'AT', 'first_shown': '2023-03...
audience_selection_approach_info    {'demographic_info': 'CRITERIA_INCLUDED', 'geo...
topic                                                                Travel & Tourism
is_funded_by_google_ad_grants                         

**OC5: Can data from an individual ad be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from a specific advertisement on the ad repository using a unique identifier, rather than relying on search terms or other parameters and filters. The assessment should review the ad repository documentation and test available features to confirm that an individual ad can be retrieved directly by its unique identifier.*



In [ ]:
# Use this cell to develop an example where a request for ads is made using an ad ID.
# Please leave the result as the output of this cell.
ad_id = None # SET THE VALUE HERE

**OC6: Can data from ads served by a specific advertiser be retrieved from the platform?**

*This item verifies whether it is possible to retrieve data from ads run by a specific advertiser, via their username or unique identifier. The assessment should review the ad repository documentation and test any available feature to retrieve data from an individual advertiser.*


In [ ]:
# Use this cell to develop an example where a request for ads from an advertiser is made.
# Please leave the result as the output of this cell.
advertiser = None # SET THE VALUE HERE

**OC7: Can ad data be retrieved from the platform using search terms?**

*This item verifies whether data on ad campaigns can be retrieved via individual or combined search terms, enabling the creation of a dataset composed of ads that mention those terms. The assessment should test search-related features to confirm that queries using keywords return relevant ad campaign data.*


In [ ]:
# Use this cell to develop an example where a request for ads is made using a search term.
# Please leave the result as the output of this cell.
search_term = None # SET THE VALUE HERE

**OC8: Does the platform use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are provided in a locale-neutral format, or, if that is not possible, whether relevant locale metadata is included. The assessment should review the ad repository documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

### COMPLETENESS

**OC9: Does the platform provide data that allows the identification of advertisers who ran ads?**

*This item verifies whether the platform discloses information on the advertisers responsible for the identified ads. The assessment should confirm whether the advertiser’s page name, URL, and unique identifier can be retrieved.*



In [9]:
# Advertiser related columns
df_creative_stats.loc[0][[
    'advertiser_id',
    'advertiser_disclosed_name',
    'advertiser_legal_name',
    'advertiser_location',
    'advertiser_verification_status'
]]

advertiser_id                     AR07993695893255618561
advertiser_disclosed_name           Bons Websistemes, SL
advertiser_legal_name               Bons Websistemes, SL
advertiser_location                                   AD
advertiser_verification_status                  VERIFIED
Name: 0, dtype: object

We can retrieve the advertiser's name and ID, but no URL

**OC10: Does the platform provide data on the funders who paid for ads?**

*This item verifies whether the platform provides data on who paid for ad campaigns. The assessment should confirm whether any sponsor information can be retrieved.*


In [12]:
df_creative_stats.loc[0][['is_funded_by_google_ad_grants', 'ad_funded_by']]

is_funded_by_google_ad_grants    False
ad_funded_by                          
Name: 0, dtype: object

In [20]:
df_creative_stats.is_funded_by_google_ad_grants.value_counts()

is_funded_by_google_ad_grants
False    100
Name: count, dtype: Int64

In [14]:
df_creative_stats.ad_funded_by.value_counts()

ad_funded_by
                                  98
Optimum media direction fz llc     1
Vivaki Mena FZ LLC                 1
Name: count, dtype: int64

There are fields regarding who paid for ad campaings, but most of the time it seems the value is an empty string. In our 100 samples, only 2 had filled values.

**OC11: Does the platform provide data on the period during which ads were served?**

*This item verifies whether the platform provides data on the days on which ad campaigns ran. The assessment should review ad records to confirm that campaigns include start and end dates (or equivalent temporal markers) covering their active period*

In [24]:
df_creative_stats.loc[0].region_stats[0]

{'region_code': 'AT',
 'first_shown': '2023-03-01',
 'last_shown': '2024-11-21',
 'times_shown_end_date': '2024-11-21',
 'times_shown_lower_bound': 0,
 'times_shown_upper_bound': 1000,
 'times_shown_start_date': '2023-05-18',
 'times_shown_availability_date': None,
 'surface_serving_stats': {'surface_serving_stats': array([{'surface': 'YOUTUBE', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'MAPS', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SEARCH', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'PLAY', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SHOPPING', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None}],
        dtype=

**OC12: Does the platform provide data on user engagement with ads?**

*This item verifies whether the platform provides data on the total number of user interactions with ad campaigns (e.g., likes, comments, shares, clicks). The assessment should review ad records to confirm that engagement metrics are available and clearly linked to each campaign.*


In [30]:
df_creative_stats.loc[0].region_stats[4]

{'region_code': 'CZ',
 'first_shown': '2023-03-01',
 'last_shown': '2024-12-27',
 'times_shown_end_date': '2024-12-27',
 'times_shown_lower_bound': 0,
 'times_shown_upper_bound': 1000,
 'times_shown_start_date': '2023-05-22',
 'times_shown_availability_date': None,
 'surface_serving_stats': {'surface_serving_stats': array([{'surface': 'PLAY', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SHOPPING', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'MAPS', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'SEARCH', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None},
         {'surface': 'YOUTUBE', 'times_shown_upper_bound': 1000, 'times_shown_lower_bound': 0, 'times_shown_availability_date': None}],
        dtype=

We can only retrieve information about the times the ads were shown, not other metrics like shares, clicks etc.

**OC13: Does the platform indicate whether ads were placed by verified or unverified advertisers?**

*This item verifies whether the platform clearly indicates whether advertisers were verified at the time their ads were served. The assessment should review ad records to confirm that a verification status field is present.*


In [2]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

### COMPLIANCE

**OC14: Does the platform flag ads that were removed due to violations of its guidelines or relevant legislation?**

*This item verifies whether the platform indicates when an ad has been moderated. At a minimum, the platform should provide the reason for removal and the date. The assessment should review ad records to confirm that moderated ads are flagged and that the corresponding moderation details are clearly documented.*



In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

**OC15: Does the platform indicate whether ad creatives were generated using artificial intelligence?**

*This item verifies whether the platform flags ad campaigns in which AI played a role in producing the content. The assessment should review ad records to confirm the presence of a field or label indicating the use of AI in ad production.*
 


In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

### CONSISTENCY

**OC23: Does the data retrieved by the API reflect what is displayed on the platform’s ad repository GUI?**

*This item verifies whether the data returned by the platform’s ad repository API corresponds to the information displayed on its GUI in all its levels of detail. It should be possible to identify in the API response information such as authorship, complete content, and other campaigning information (e.g., amount spent, impressions reached). The assessment should compare API responses with the GUI to confirm that key elements—such as authorship, full content, and campaign information (e.g., spending, impressions)—are consistently included.*


In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

**OC24: Are the results returned by the platform consistently reproducible?**


*This item verifies whether data accessed and extracted via the platform’s ad repository at a given time is consistent with other collections performed similarly, including cases where content was deleted in the interim. The assessment should perform repeated queries to confirm reproducibility of results.*

Test instructions: Please, develop a test that collects ads for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [ ]:
# Use this cell, or more, to develop a process that collects ads more than one time using the same request parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:
total_runs = 5 
for idx in range(total_runs):
    index = idx + 1
    filepath = f"google-ads-question-24-run-{index}-{datetime.now().isoformat()}"

    # DEVELOP YOUR CODE HERE
    response = None # CHANGE VALUE

    with open(filepath, "w") as fout:
        json.dump(response, fout)

**OC25: Is the data returned by the platform consistent with the parameters and filters used in the request?**

*This item verifies whether the data retrieved through the ad repository accurately reflects the parameters and filters specified at the time of the request. The assessment should run test queries with different filters to confirm that results consistently match the requested conditions.*

Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [ ]:
# Use this cell, or more, to develop a process that collects ads more than one time using different parameters.
# Describe the endpoint and the search parameters used.
# Please add each response for each time a request is made in a file and save it in the data folder of this repository.
# The files naming pattern should be as stated below:
parameters = [] # SET A LIST WITH A LEAST 5 PARAMETERS
for idx, param in parameters:
    index = idx + 1 
    filepath = f"google-ads-question-25-run-{index}-{param}-{datetime.now().isoformat()}"
    
    # DEVELOP YOUR CODE HERE
    response = None # CHANGE VALUE

    with open(filepath, "w") as fout:
        json.dump(response, fout)

### RELEVANCE

**OC26: Does the platform allow the use of temporal filters to retrieve data on ads?**

*This item verifies whether the ad repository allows filtering ad campaign data based on the time period during which the ads were served. The assessment should test queries with temporal filters to confirm that results accurately reflect the specified date ranges.*



In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

**OC27: Does the platform allow filtering advertising data by ad category?**

*This item verifies whether the ad repository allows filtering ad campaign data based on any possible categories assigned at the time of ad creation. The assessment should run test queries with category filters to confirm that results align with the selected classifications.*


In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

**OC28: Does the platform allow filtering advertising data by geographic location?**

*This item verifies whether the platform allows filtering data on ad campaigns by one or more geographic locations. The assessment should test queries with location filters to confirm that results match the specified areas.*

In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

### ACCURACY

**OC29: Does the platform provide age and gender data on the audiences of ads?**

*This item verifies whether the platform provides data on the age and gender of audiences reached. The assessment should review the ad records to confirm that these breakdowns are available and consistently reported*

In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

**OC30: Does the platform provide subnational geographic data on the audience reached by ads?**

*This item verifies whether the platform provides data on the subnational geographic location of audiences reached. The assessment should review the ad records to confirm that such location breakdowns are available and consistently reported.*

In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

**OC31: Does the platform include data on audience targeting criteria defined by advertisers?**

*This item verifies whether the platform provides data on audience targeting criteria defined by the advertiser when publishing ads (e.g., demographic and geographic segments, as well as interests, attitudes, behaviors, and keywords). The assessment should review ad records to confirm that these targeting parameters are available and consistently reported.*


In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

**OC32: Does the platform provide granular volume ranges for ad impressions?**

*This item verifies whether the ad repository provides impression values for ads, using ranges that closely approximate the actual numbers. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to 1,000 impressions should be displayed in intervals no larger than 100; between 1,000 and 10,000 in intervals no larger than 1,000; between 10,000 and 100,000 in intervals no larger than 10,000; between 100,000 and 1 million or above, in intervals no larger than 100,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.

**OC33: Does the platform provide granular investment ranges for ad spending?**

*This item verifies whether the ad repository provides spending values for ads, using ranges that closely approximate the actual amounts. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to $100 should be displayed in intervals no larger than $10; between $100 and $1,000 in intervals no larger than $100; and between $1,000 and $10,000 in intervals no larger than $1,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces.*


In [ ]:
# Use this cell to develop an example where a request for ads is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.